# Speech-to-Speech test bench

Use this notebook to test the complete local path: **microphone (or a WAV file) → STT WebSocket service → response text → Chhattisgarhi VITS TTS → audio playback**.

Run the cells in order. The first STT/TTS invocation can take a while because models are loaded into memory.

## One-time environment setup

The notebook kernel needs the STT dependencies, Coqui TTS, and a Jupyter kernel. Uncomment and run the appropriate line below only if an import later reports that a package is missing. Run the notebook with the same Python environment used for the STT service.

In [ ]:
# %pip install -r STT/stt-service/requirements.txt
# %pip install TTS ipykernel
# Restart the kernel after installing packages.

In [18]:
from __future__ import annotations

import asyncio
import json
import os
import subprocess
import sys
import time
import uuid
import wave
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

import numpy as np
from IPython.display import Audio, display

# The notebook is intended to live at the repository root.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'STT' / 'stt-service').exists():
    raise RuntimeError('Start Jupyter from the repository root (the folder containing STT/ and TTS/).')

STT_ROOT = REPO_ROOT / 'STT' / 'stt-service'
TTS_ROOT = REPO_ROOT / 'TTS' / 'chattisgarhi-tts-models'
OUTPUT_DIR = REPO_ROOT / 'speech2speech_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

SAMPLE_RATE = 16_000             # fixed STT input contract
CHUNK_MS = 40                    # same cadence as a live microphone client
STT_HTTP_URL = 'http://127.0.0.1:8000/health'
STT_WS_URL = 'ws://127.0.0.1:8000/ws/stt'

print(f'Repository: {REPO_ROOT}')
print(f'Outputs:    {OUTPUT_DIR}')

Repository: /run/media/rtx/Files/Study/Semester 5/Minor/code
Outputs:    /run/media/rtx/Files/Study/Semester 5/Minor/code/speech2speech_outputs


## Start the STT service

This reuses an already-running service if one is available; otherwise it starts `STT/stt-service/run.py` in the background. Set `STT_LANGUAGE=hne` in `STT/stt-service/.env` before this cell if you want the MMS Chhattisgarhi recognizer.

In [38]:
def stt_health():
    """Return the health payload, or None while the local service is unavailable."""
    try:
        with urlopen(STT_HTTP_URL, timeout=1) as response:
            return json.load(response)
    except (URLError, TimeoutError, OSError):
        return None

health = stt_health()
if health is None:
    # Keep this handle in the kernel so `stop_stt_service()` can stop only the process this notebook started.
    stt_process = subprocess.Popen([sys.executable, 'run.py'], cwd=STT_ROOT)
    deadline = time.monotonic() + 300  # MMS can take substantial time to load on first use.
    while time.monotonic() < deadline:
        time.sleep(1)
        health = stt_health()
        if health is not None:
            break
    if health is None:
        raise RuntimeError('STT service did not become ready within 5 minutes. Check the cell output above.')

print('STT is ready:', health)

INFO:     Started server process [47543]
INFO:     Waiting for application startup.
2026-09-04 07:35:11,363 [INFO] stt-server: Loading ASR model 'base' on cuda (int8)...
2026-09-04 07:35:16,012 [WARNING] stt-server: STT_API_KEY is not set — server will accept unauthenticated connections. Fine for localhost dev, NOT fine once this is reachable from the internet.
2026-09-04 07:35:16,012 [INFO] stt-server: Model loaded. Capacity: 8 concurrent calls. Server ready.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


STT is ready:INFO:     127.0.0.1:39784 - "GET /health HTTP/1.1" 200 OK
 {'status': 'ok', 'model': 'base', 'device': 'cuda', 'active_sessions': 0, 'max_sessions': 8, 'at_capacity': False}


In [25]:
import torch
from TTS.utils.synthesizer import Synthesizer

# Change this to 'Male' to use the male Chhattisgarhi voice.
VOICE = 'Female'
TTS_DEVICE_IS_CUDA = torch.cuda.is_available()

tts = Synthesizer(
    tts_checkpoint=str(TTS_ROOT / VOICE / 'best_model.pth'),
    tts_config_path=str(TTS_ROOT / VOICE / 'config.json'),
    use_cuda=TTS_DEVICE_IS_CUDA,
)
print(f'TTS ready: {VOICE} voice on {"CUDA" if TTS_DEVICE_IS_CUDA else "CPU"}.')

 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
TTS ready: Female voice on CUDA.


## Provide speech input

Use either option below. Recording produces the exact 16 kHz mono PCM16 format required by the STT service. For a WAV file, the helper resamples and converts it when `soundfile` is installed.

In [ ]:
def record_from_microphone(seconds: float = 6.0) -> np.ndarray:
    """Record mono 16 kHz PCM16 from the default microphone for a bounded test turn."""
    import sounddevice as sd

    frames = int(seconds * SAMPLE_RATE)
    print(f'Recording for {seconds:.1f}s. Speak, then leave a brief pause at the end…')
    audio = sd.rec(frames, samplerate=SAMPLE_RATE, channels=1, dtype='int16')
    sd.wait()
    print('Recording complete.')
    return np.ascontiguousarray(audio[:, 0])

# Adjust the duration, run this cell, then run the pipeline cell.
input_pcm16 = record_from_microphone(seconds=6.0)
display(Audio(input_pcm16, rate=SAMPLE_RATE))

Recording for 6.0s. Speak, then leave a brief pause at the end…
Recording complete.


In [34]:
def load_wav_as_pcm16(path: str | Path) -> np.ndarray:
    """Load a WAV/audio file and return contiguous mono 16 kHz PCM16 for STT."""
    import soundfile as sf

    audio, source_rate = sf.read(path, dtype='float32', always_2d=True)
    audio = audio.mean(axis=1)  # downmix stereo safely
    if source_rate != SAMPLE_RATE:
        # scipy is deliberately imported only when resampling is needed.
        from scipy.signal import resample_poly
        from math import gcd
        divisor = gcd(source_rate, SAMPLE_RATE)
        audio = resample_poly(audio, SAMPLE_RATE // divisor, source_rate // divisor)
    return np.ascontiguousarray(np.clip(audio, -1, 1) * 32767, dtype=np.int16)

# Example: input_pcm16 = load_wav_as_pcm16('path/to/your_recording.wav')
# display(Audio(input_pcm16, rate=SAMPLE_RATE))

## Run the complete Speech → Speech pipeline

The STT client streams the recorded audio in real-time-sized chunks and collects every server event. Only `final` transcripts are passed onward, following the STT integration contract. Replace `make_response()` with an LLM or your business logic when it is ready; its return value should be text for the Chhattisgarhi TTS model.

In [36]:
async def transcribe_pcm16(pcm16: np.ndarray, *, realtime: bool = True) -> list[dict]:
    """Stream PCM16 to STT and return all protocol events, including the final transcript."""
    import websockets

    if pcm16.dtype != np.int16 or pcm16.ndim != 1:
        raise ValueError('Expected a one-dimensional int16 PCM array at 16 kHz.')

    events: list[dict] = []
    session_id = f'notebook-{uuid.uuid4().hex[:8]}'
    chunk_samples = SAMPLE_RATE * CHUNK_MS // 1000

    async with websockets.connect(f'{STT_WS_URL}/{session_id}', ping_interval=None) as websocket:
        async def receive_events():
            async for message in websocket:
                event = json.loads(message)
                events.append(event)
                label = event['type'].upper()
                if event.get('text'):
                    print(f'[{label}] {event["text"]}')
                else:
                    print(f'[{label}]')

        receiver = asyncio.create_task(receive_events())
        for start in range(0, len(pcm16), chunk_samples):
            await websocket.send(pcm16[start:start + chunk_samples].tobytes())
            if realtime:
                await asyncio.sleep(CHUNK_MS / 1000)
        await websocket.send(json.dumps({'action': 'stop'}))
        await receiver  # server closes after flushing the last final event

    return events

def make_response(transcript: str) -> str:
    """Provide a small local response layer; replace this with your LLM/orchestration call."""
    normalized = transcript.strip().lower()
    if not normalized:
        return 'माफ करना, मोला कुछ सुनाई नहीं दिस। फेर ले बोलव।'
    if any(word in normalized for word in ('नमस्ते', 'hello', 'hi')):
        return 'नमस्ते! मैं आप मन के मदद बर तैयार हवंव।'
    # Echoing makes an end-to-end test useful even before an LLM is connected.
    return f'आपने कहा: {transcript}'

def synthesize_response(text: str, *, length_scale: float = 1.0) -> Path:
    """Synthesize a response and save it as a timestamped WAV file for playback and inspection."""
    tts.tts_model.length_scale = length_scale  # Coqui's supported speed control
    wav = tts.tts(text=text)
    output_path = OUTPUT_DIR / f'response_{time.strftime("%Y%m%d_%H%M%S")}.wav'
    tts.save_wav(wav=wav, path=str(output_path))
    return output_path


In [39]:
# `input_pcm16` must come from the microphone cell or `load_wav_as_pcm16()`.
events = await transcribe_pcm16(input_pcm16, realtime=True)
final_transcripts = [event['text'] for event in events if event['type'] == 'final' and event.get('text')]
transcript = final_transcripts[-1] if final_transcripts else ''

print('\nFinal transcript:', transcript or '(none)')
response_text = make_response(transcript)
print('TTS response:', response_text)

response_wav = synthesize_response(response_text, length_scale=1.0)
print('Saved:', response_wav)
display(Audio(filename=str(response_wav), autoplay=True))

INFO:     ('127.0.0.1', 39796) - "WebSocket /ws/stt/notebook-e50ea43f" [accepted]
2026-09-04 07:35:16,929 [INFO] stt-server: [notebook-e50ea43f] connected (1/8 active)
INFO:     connection open


[SESSION_STARTED]
[SPEECH_STARTED]
[PARTIAL] मस ए
[PARTIAL] मौस बात मत करी
[PARTIAL] मौसै बात मत करे
[FINAL] मौसै बात मत करे


2026-09-04 07:35:23,030 [INFO] stt-server: [notebook-e50ea43f] session closed — duration=6.1s, utterances=1, (0/8 active)
INFO:     Shutting down
INFO:     connection closed


CancelledError: 

## TTS-only sanity check

Use this to validate a voice/model independently of STT. `length_scale > 1` speaks more slowly; use the `length_scale` property rather than a `speed=` argument.

In [ ]:
manual_text = 'नमस्ते! मैं आप मन के मदद बर तैयार हवंव।'
manual_wav = synthesize_response(manual_text, length_scale=1.0)
print('Saved:', manual_wav)
display(Audio(filename=str(manual_wav), autoplay=False))

 > Text splitted to sentences.
['नमस्ते!', 'मैं आप मन के मदद बर तैयार हवंव।']
 > Processing time: 0.8383347988128662
 > Real-time factor: 0.2269803820459688
Saved: /run/media/rtx/Files/Study/Semester 5/Minor/code/speech2speech_outputs/response_20260904_073029.wav


In [24]:
def stop_stt_service():
    """Stop only a service process launched by this notebook; leave externally started servers alone."""
    process = globals().get('stt_process')
    if process is not None and process.poll() is None:
        process.terminate()
        process.wait(timeout=10)
        print('Stopped the STT process launched by this notebook.')
    else:
        print('No STT process launched by this notebook is running.')

# Call `stop_stt_service()` when you are finished, if this notebook started the server.